# CPSC 483 Final Project
## Breast Cancer Diagnosis Prediction Using Machine Learning
## 1. Problem Statement & Introduction

The goal of this project is to build a machine learning model that can classify tumors as either malignant (cancerous) or benign (non-cancerous).

The dataset contains diagnostic measurements computed from digitized images of breast masses.

Features include:
- Mean radius
- Mean texture
- Mean perimeter
- Mean area
- Mean smoothness
- Additional diagnostic tumor measurements

This is a **binary classification problem**, where:
- 0 = malignant
- 1 = benign

We are going to compare logistic regression and random forest to see which model performs the best.

## 2. Data Loading & Exploration

In this section, we load the dataset and convert it into a pandas DataFrame for easier analysis and manipulation.

In [ ]:
# Import dataset and pandas
from sklearn.datasets import load_breast_cancer
import pandas as pd

# Load dataset
data = load_breast_cancer()

# Convert to DataFrame
df = pd.DataFrame(
    data.data,
    columns=data.feature_names
)

# Add target column (labels)
df["target"] = data.target

# Display first 5 rows
df.head()

## 3. Data Preparation

We separate the dataset into features, which are the input variables, and labels, which is the target we want to predict. Afterwards, we separated the data into sets for training and testing.

In [ ]:
from sklearn.model_selection import train_test_split

# X is for the features and y is for the labels
X = data.data
y = data.target

# Split the data: 80% for training and 20% for testing
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42
)

## 4. Model 1 – Logistic Regression

Logistic Regression is a simple and efficient model for binary classification problems.
We use it as a baseline model to compare performance.

In [ ]:
from sklearn.linear_model import LogisticRegression

# Create the logistic regression model
lr_model = LogisticRegression(max_iter=5000)

# Train the logistic regression model
lr_model.fit(X_train, y_train)

# Evaluate accuracy of the logistic regression model
print("Logistic Regression Accuracy:", lr_model.score(X_test, y_test))

## 5. Model 2 – Random Forest

Random Forest is a model that puts together several decision trees for a single result. It is more powerful than simpler models such as logistic regression.

In [ ]:
from sklearn.ensemble import RandomForestClassifier

# Create the random forest model
rf_model = RandomForestClassifier(
    random_state=42)

# Train the random forest model
rf_model.fit(X_train, y_train)

# Evaluate accuracy of the random forest model
print("Random Forest Accuracy:",
rf_model.score(X_test, y_test))

### 5.1 Random Forest Hyperparameter Tuning

Because the dataset is relatively small and Random Forest models can overfit, we used RandomizedSearchCV to tune the model’s hyperparameters.

In [ ]:
from sklearn.model_selection import RandomizedSearchCV
from sklearn.ensemble import RandomForestClassifier

# Define parameter options for tuning
param_grid = {
    "n_estimators": [50, 100, 200, 300],
    "max_depth": [None, 5, 10, 20],
    "min_samples_split": [2, 5, 10],
    "min_samples_leaf": [1, 2, 4],
    "max_features": ["sqrt", "log2"]
}

# Create base Random Forest model
rf_base = RandomForestClassifier(random_state=42)

# Randomized Search for best hyperparameters
random_search = RandomizedSearchCV(
    estimator=rf_base,
    param_distributions=param_grid,
    n_iter=20,
    cv=5,
    scoring="accuracy",
    random_state=42,
    n_jobs=1
)

# Fit RandomizedSearchCV on training data
random_search.fit(X_train, y_train)

# Show best parameters and best cross-validation score
print("Best Parameters:", random_search.best_params_)
print("Best Cross-Validation Accuracy:", random_search.best_score_)

# Use the best model
rf_model = random_search.best_estimator_

# Evaluate tuned model on test data
print("Tuned Random Forest Test Accuracy:", rf_model.score(X_test, y_test))

## 6. Model Evaluation

We evaluate the random forest model using accuracy, precision, recall, and F1-score.

In [ ]:
from sklearn.metrics import classification_report

# Uses random forest model to make predictions on whether or not a tumor is cancerous
predictions = rf_model.predict(X_test)

# Print evaluation metrics
print(
classification_report(
    y_test,
    predictions
))

## 7. Confusion Matrix

The confusion matrix shows correct predictions vs incorrect predictions. This helps visualize the random forest model’s performance more clearly.

In [ ]:
from sklearn.metrics import ConfusionMatrixDisplay

# Display confusion matrix
ConfusionMatrixDisplay.from_estimator(
    rf_model,
    X_test,
    y_test
)

## 8. Feature Importance

We analyze which features are most important for the model's predictions. This gives us a better understanding on which factors influence the outcome.

In [ ]:
# Get feature importance scores from random forest model
importance = pd.Series(
rf_model.feature_importances_,
index=data.feature_names
).sort_values(ascending=False)

# Show the top 10 features from the random forest model
print(importance.head(10))

## 9. Feature Importance Visualization

We visualize the top features using a bar chart for better understanding.

In [ ]:
import matplotlib.pyplot as plt

# Plot top 10 features from random forest model
importance.head(10).plot(kind="bar")

plt.title("Top Feature Importances")
plt.xlabel("Features")
plt.ylabel("Importance Score")

plt.show()

## 10. Conclusion

This project compared Logistic Regression and Random Forest models for predicting breast cancer.

After applying RandomizedSearchCV, the Random Forest model was tuned to reduce the risk of overfitting and improve reliability for the small dataset.

Although both models performed well, Random Forest had the strongest performance with an accuracy of  97.37%.

These results show that machine learning can support early disease classification and demonstrates that predictive models can be very useful in healthcare settings.